# llm-request SDK 종합 검증 노트북 (`verify_all.ipynb`)

`llm-request` SDK의 **모든 공개 함수 × 모든 분기**를 로컬 환경에서 실행·검증한다.

## 선행 조건

다음이 모두 실행 중이어야 한다 (file-manager·llm-request는 localhost, 나머지는 192.7.9.45):

| 서비스 | 포트 | 실행 방법 |
|--------|------|----------|
| Redis | 6379 | `docker compose -f infra/docker-compose.yml up -d redis` |
| MongoDB | 27017 | `docker compose -f infra/docker-compose.yml up -d mongodb` |
| llm-handler | 8100 | `docker compose --profile handlers up -d llm-handler` |
| mdb-handler | 8300 | `docker compose --profile handlers up -d mdb-handler` |
| file-manager | 8102 | `cd packages/file-manager/service && uv run uvicorn app.main:app --port 8102` |
| llm-request | 8101 | `cd packages/llm-request/service && uv run uvicorn app.main:app --port 8101` |

`.env`에 `GEMINI_API_KEY` 필수.

## 검증 범위

- 공개 함수 6종: `llm_request`, `llm_request_async`, `llm_request_stream`, `llm_request_stream_async`, `done_check`, `fetch_by_cache_key`
- 플래그: `cache_mode`, `save_mode`, `debug`, `ttl_seconds`, `context_cache_ttl`
- 파일 분기: inline 단일/복수, Files API, 부분 섞기, 20MB 한도, PDF split, context_cache
- 오류 경로: 지원 안 되는 mime 타입
- 동시성: single-flight dedup

## 0. 환경 설정

In [1]:
import os
import nest_asyncio

nest_asyncio.apply()

os.environ["LLM_REQUEST_URL"]  = "http://localhost:8101"
os.environ["MONGODB_DATABASE"] = "kw_ai_dev"  # 테스트 전용 DB

TEST_DB = os.environ["MONGODB_DATABASE"]

# SDK 기본 http_timeout이 600초로 상향됐으므로 별도 설정 불필요.
# 필요 시 요청별로 timings={"http_timeout": 1200} 같이 오버라이드.

print(f"LLM_REQUEST_URL  = {os.environ['LLM_REQUEST_URL']}")
print(f"MONGODB_DATABASE = {TEST_DB}")

LLM_REQUEST_URL  = http://localhost:8101
MONGODB_DATABASE = kw_ai_dev


## 1. 공통 유틸리티

In [2]:
import time
import uuid
from contextlib import contextmanager
from pathlib import Path


@contextmanager
def timer(label: str):
    start = time.perf_counter()
    yield
    elapsed_ms = (time.perf_counter() - start) * 1000
    print(f"  [{label}] {elapsed_ms:.0f}ms")


def print_meta(title: str, meta: dict) -> None:
    print(f"--- {title} ---")
    print(f"  source        = {meta.get('source')}")
    print(f"  cache_hit     = {meta.get('cache_hit')}")
    print(f"  cache_key     = {str(meta.get('cache_key',''))[:16]}...")
    if meta.get('route_reason'):
        print(f"  route_reason  = {meta['route_reason']}")
    if meta.get('file_route'):
        print(f"  file_route    = {meta['file_route']}")


def unique_q(base: str) -> str:
    """캐시 미스를 강제하기 위해 매번 다른 입력 생성."""
    return f"{base} [id={uuid.uuid4().hex[:8]}]"

In [3]:
import mimetypes
from llm_request import FileInput


def build_file_input(path) -> FileInput:
    """로컬 파일 → FileInput (meta 5필드 + bytes).

    file-manager 등록은 서비스가 내부에서 처리하므로 에이전트는
    로컬에서 FileInput만 만들면 된다.
    """
    p = Path(path)
    return FileInput(
        file_name=p.name,
        file_size=p.stat().st_size,
        file_mtime=int(p.stat().st_mtime),
        mime_type=mimetypes.guess_type(p.name)[0] or "application/octet-stream",
        extension=p.suffix.lstrip("."),
        data=p.read_bytes(),
    )

## 2. 서비스 헬스체크

모든 서비스가 올라와 있는지 확인. 하나라도 FAIL이면 이후 셀은 실패.

In [4]:
import httpx

SERVICES = [
    ("llm-request",  "http://localhost:8101/health"),
    ("file-manager", "http://localhost:8102/health"),
    ("llm-handler",  "http://localhost:8100/health"),
    ("mdb-handler",  "http://192.7.9.45:8300/health"),
]

async with httpx.AsyncClient(timeout=5.0) as c:
    for name, url in SERVICES:
        try:
            r = await c.get(url)
            ok = r.status_code == 200
            print(f"  [{'OK' if ok else 'FAIL'}] {name:<13} -> {r.status_code}")
        except Exception as e:
            print(f"  [FAIL] {name:<13} -> {type(e).__name__}: {e}")

  [OK] llm-request   -> 200
  [OK] file-manager  -> 200
  [OK] llm-handler   -> 200
  [OK] mdb-handler   -> 200


## 3. `llm_request_async` — 기본 (캐시 miss → hit)

동일 입력 2회 호출해 첫 번째는 LLM, 두 번째는 MDB 캐시 히트가 되는지 확인.

In [5]:
from llm_request import llm_request_async

Q1 = unique_q("대한민국의 수도는?")

with timer("1회차 (cache miss → LLM)"):
    r1, m1 = await llm_request_async(
        {"contents": Q1, "model": "gemini-2.5-flash"},
        database=TEST_DB,
    )
print_meta("1회차", m1)
print(f"  text          = {r1.get('text','')[:80]}")

with timer("2회차 (cache hit)"):
    r2, m2 = await llm_request_async(
        {"contents": Q1, "model": "gemini-2.5-flash"},
        database=TEST_DB,
    )
print_meta("2회차", m2)

assert m2.get("cache_hit") is True, "2회차는 캐시 히트여야 한다"
assert m1.get("cache_key") == m2.get("cache_key"), "동일 입력 → 동일 cache_key"
print("\n  OK: miss→hit 동작 확인")

  [1회차 (cache miss → LLM)] 1298ms
--- 1회차 ---
  source        = live
  cache_hit     = False
  cache_key     = 00b82a30ff5dad80...
  text          = 대한민국의 수도는 **서울**입니다.
  [2회차 (cache hit)] 10ms
--- 2회차 ---
  source        = cache
  cache_hit     = True
  cache_key     = 00b82a30ff5dad80...

  OK: miss→hit 동작 확인


## 4. `llm_request` (동기 래퍼)

내부에서 `asyncio.run`을 호출하지만 `nest_asyncio`로 중첩 이벤트 루프를 허용했으므로 직접 실행 가능.

In [6]:
from llm_request import llm_request

r, m = llm_request(
    {"contents": unique_q("2+2는?"), "model": "gemini-2.5-flash"},
    database=TEST_DB,
)

print_meta("sync", m)
print(f"  text          = {r.get('text','')[:80]}")

--- sync ---
  source        = live
  cache_hit     = False
  cache_key     = 7c24168860fb7998...
  text          = 4


## 5. 플래그 — `cache_mode` / `save_mode` / `debug` / `ttl_seconds`

- `cache_mode=False`: 캐시 조회 스킵 → 항상 LLM 호출
- `save_mode=False`: 응답 저장 안 함 → 이후 done_check 시 miss
- `debug=True`: meta에 `route_reason` 포함
- `ttl_seconds`: MDB TTL 지정

In [7]:
# cache_mode=False : 첫 호출로 저장된 결과가 있어도 무시하고 LLM 호출
Q_CM = unique_q("캐시 우회 테스트")
await llm_request_async({"contents": Q_CM, "model": "gemini-2.5-flash"}, database=TEST_DB)
r, m = await llm_request_async(
    {"contents": Q_CM, "model": "gemini-2.5-flash"},
    cache_mode=False,
    database=TEST_DB,
)
print_meta("cache_mode=False (두번째 호출도 LLM)", m)
assert m.get("cache_hit") is False
print("  OK: cache_mode=False는 캐시 히트 안 됨")

--- cache_mode=False (두번째 호출도 LLM) ---
  source        = live
  cache_hit     = False
  cache_key     = a89b67ec8785382d...
  OK: cache_mode=False는 캐시 히트 안 됨


In [8]:
# save_mode=False : 호출은 하지만 저장은 안 함 → done_check 시 miss
from llm_request import done_check

Q_SM = unique_q("저장 스킵 테스트")
r, m = await llm_request_async(
    {"contents": Q_SM, "model": "gemini-2.5-flash"},
    save_mode=False,
    database=TEST_DB,
)
print_meta("save_mode=False", m)

dc = await done_check({"contents": Q_SM, "model": "gemini-2.5-flash"}, database=TEST_DB)
print(f"  done_check.cache_hit = {dc['cache_hit']}")
assert dc["cache_hit"] is False, "save_mode=False 이후 done_check는 miss여야 한다"
print("  OK: save_mode=False는 저장 안 함")

--- save_mode=False ---
  source        = live
  cache_hit     = False
  cache_key     = bc4fe436c392ef27...
  done_check.cache_hit = False
  OK: save_mode=False는 저장 안 함


In [9]:
# debug=True : meta.route_reason 확인
r, m = await llm_request_async(
    {"contents": unique_q("debug 경로"), "model": "gemini-2.5-flash"},
    debug=True,
    database=TEST_DB,
)
print_meta("debug=True", m)
assert m.get("route_reason") is not None, "debug=True일 때 route_reason 포함돼야 함"
print("  OK: route_reason 포함 확인")

--- debug=True ---
  source        = live
  cache_hit     = False
  cache_key     = 6af9238a2b229e1d...
  route_reason  = cache_miss
  OK: route_reason 포함 확인


In [10]:
# mdb_ttl : MDB TTL 지정 (저장은 되지만 주기적으로 만료)
r, m = await llm_request_async(
    {"contents": unique_q("TTL 테스트"), "model": "gemini-2.5-flash"},
    timings={"mdb_ttl": 120},
    database=TEST_DB,
)
print_meta("timings={'mdb_ttl': 120}", m)

--- timings={'mdb_ttl': 120} ---
  source        = live
  cache_hit     = False
  cache_key     = 7ca6d22fcc480e32...


## 6. `done_check` + `fetch_by_cache_key`

In [11]:
from llm_request import fetch_by_cache_key

Q_F = unique_q("페치 테스트 질문")
await llm_request_async({"contents": Q_F, "model": "gemini-2.5-flash"}, database=TEST_DB)

dc = await done_check({"contents": Q_F, "model": "gemini-2.5-flash"}, database=TEST_DB)
print(f"done_check: cache_key={dc['cache_key'][:16]}... cache_hit={dc['cache_hit']}")
assert dc["cache_hit"] is True

r, m = await fetch_by_cache_key(dc["cache_key"], database=TEST_DB)
print_meta("fetch_by_cache_key", m)
print(f"  text = {r.get('text','')[:80]}")

done_check: cache_key=b25c7a79c7d6d33f... cache_hit=True
--- fetch_by_cache_key ---
  source        = cache
  cache_hit     = True
  cache_key     = b25c7a79c7d6d33f...
  text = Fetch API를 사용하는 코드를 테스트하는 것은 웹 애플리케이션 개발에서 매우 중요합니다. 실제 네트워크 요청은 느리고, 불안정하며, 테스트


## 7. `llm_request_stream_async` — 스트리밍

- 1회차 (miss): SSE 청크 여러 개
- 2회차 (hit): 캐시 결과가 단일 청크로 래핑되어 동일 인터페이스로 반환

In [12]:
from llm_request import llm_request_stream_async

Q_S = unique_q("스트리밍으로 답해줘: 한국 전통 악기 3가지")

chunks_1 = []
async for ev in llm_request_stream_async(
    {"contents": Q_S, "model": "gemini-2.5-flash"},
    database=TEST_DB,
):
    chunks_1.append(ev)

print(f"1회차: {len(chunks_1)}개 청크")
full_text = "".join(c.get("text", "") for c in chunks_1)
print(f"  전체 텍스트: {full_text[:120]}")

chunks_2 = []
async for ev in llm_request_stream_async(
    {"contents": Q_S, "model": "gemini-2.5-flash"},
    database=TEST_DB,
):
    chunks_2.append(ev)

print(f"\n2회차 (cache hit): {len(chunks_2)}개 청크")

1회차: 9개 청크
  전체 텍스트: 안녕하세요, 스트리머입니다! ✨ 바로 질문 들어왔네요! 한국 전통 악기 3가지, 지금 바로 알려드릴게요!

자, 집중! 바로 갑니다! 🎤

1.  **가야금 (Gayageum)**
    *   **설명:** 12줄

2회차 (cache hit): 1개 청크


## 8. `llm_request_stream` (동기 스트리밍)

In [13]:
from llm_request import llm_request_stream

Q_SS = unique_q("동기 스트리밍으로 답해줘: 한국 5대 명절 각 2~3줄 설명")

events_1 = list(llm_request_stream(
    {"contents": Q_SS, "model": "gemini-2.5-flash"},
    database=TEST_DB,
))
print(f"1회차 (miss): {len(events_1)}개 청크")

events_2 = list(llm_request_stream(
    {"contents": Q_SS, "model": "gemini-2.5-flash"},
    database=TEST_DB,
))
print(f"2회차 (cache hit): {len(events_2)}개 청크")

assert len(events_1) > 1, "1회차는 SSE 청크가 여러 개여야 miss 검증 가능"
assert len(events_2) == 1, "2회차는 단일 청크로 래핑됨 (hit)"
print("  OK: miss→hit 청크 수 차이 확인")

1회차 (miss): 13개 청크
2회차 (cache hit): 1개 청크
  OK: miss→hit 청크 수 차이 확인


## 9. 파일 준비 — `build_file_input`

테스트 파일들을 로컬에서 `FileInput`으로 구성. file-manager 호출은 서비스가 내부에서 처리.

In [14]:
SAMPLES_DIR = Path(".").resolve()

FILES = {
    "small_pdf":  SAMPLES_DIR / "small_file.pdf",    # 612KB
    "medium_pdf": SAMPLES_DIR / "medium_file.pdf",   # 24MB
    "large_pdf":  SAMPLES_DIR / "large_file.pdf",    # 92MB
    "image":      SAMPLES_DIR / "test_image.png",    # 1.9MB
    "text":       SAMPLES_DIR / "sample.txt",        # 2KB
    "audio":      SAMPLES_DIR / "test_audio.m4a",    # 4MB (mime 미지원 테스트용)
}

inputs: dict[str, FileInput] = {}
for key, path in FILES.items():
    if not path.exists():
        print(f"  [SKIP] {key}: {path} 없음")
        continue
    fi = build_file_input(path)
    inputs[key] = fi
    print(f"  [OK]   {key:<10} mime={fi.mime_type:<18} size={fi.file_size:>10,}B")

  [OK]   small_pdf  mime=application/pdf    size=   612,269B
  [OK]   medium_pdf mime=application/pdf    size=24,770,651B
  [OK]   large_pdf  mime=application/pdf    size=92,716,371B
  [OK]   image      mime=image/png          size= 1,914,348B
  [OK]   text       mime=text/plain         size=     1,850B
  [OK]   audio      mime=audio/mp4          size= 4,006,244B


## 10. Inline — PDF 단일 (612KB)

PDF 한 개가 50MB 미만이므로 inline 분기.

In [15]:
r, m = await llm_request_async(
    {"contents": unique_q("이 문서를 한 문장으로 요약해줘"), "model": "gemini-2.5-flash"},
    files=[inputs["small_pdf"]],
    database=TEST_DB,
    debug=True,
)
print_meta("small_pdf (inline)", m)
print(f"  text = {r.get('text','')[:150]}")

--- small_pdf (inline) ---
  source        = live
  cache_hit     = False
  cache_key     = 7f0e4343035fce73...
  route_reason  = cache_miss
  text = 이 문서는 KB금융그룹이 2024년 투자자 관점에서 작성한 지속가능경영보고서로, 리스크 관리, 기후위기 대응, 포용금융, 금융소비자보호, 디지털 혁신 및 기술 등 주요 지속가능성 관련 위험과 기회에 대한 그룹의 전략과 성과를 담고 있습니다.


## 11. Inline — 이미지

In [16]:
r, m = await llm_request_async(
    {"contents": unique_q("이 이미지에 무엇이 있나?"), "model": "gemini-2.5-flash"},
    files=[inputs["image"]],
    database=TEST_DB,
    debug=True,
)
print_meta("image (inline)", m)
print(f"  text = {r.get('text','')[:150]}")

--- image (inline) ---
  source        = live
  cache_hit     = False
  cache_key     = 4de34d76bcebb089...
  route_reason  = cache_miss
  text = 이 이미지는 신비롭고 고요한 밤의 숲 풍경을 담고 있습니다. 주요 특징들은 다음과 같습니다:

*   **전반적인 분위기:** 짙은 안개나 박무가 깔려 있어 몽환적이고 신비로운 느낌을 줍니다. 전체적으로 차분한 푸른색과 녹색 계열의 색감이 지배적이며, 달빛과 발광하는 


## 12. Inline — 텍스트

In [17]:
r, m = await llm_request_async(
    {"contents": unique_q("이 텍스트 파일의 주제는?"), "model": "gemini-2.5-flash"},
    files=[inputs["text"]],
    database=TEST_DB,
    debug=True,
)
print_meta("sample.txt (inline)", m)
print(f"  text = {r.get('text','')[:150]}")

--- sample.txt (inline) ---
  source        = live
  cache_hit     = False
  cache_key     = dbb3f1a159b6a11d...
  route_reason  = cache_miss
  text = 이 텍스트 파일의 주제는 "짧은 텍스트 컨텐츠"입니다.


## 13. Files API — 대용량 PDF (92MB, 자동 split)

PDF 한 개가 50MB를 초과하면 Files API 분기 + PDF split. 첫 호출은 업로드 포함이라 분 단위로 걸릴 수 있음.

In [18]:
if "large_pdf" not in inputs:
    print("  [SKIP] large_file.pdf 없음")
else:
    with timer("large_pdf 첫 호출 (업로드+split+LLM)"):
        r, m = await llm_request_async(
            {"contents": unique_q("이 PDF의 제목 한 줄"), "model": "gemini-2.5-flash", "max_tokens": 100},
            files=[inputs["large_pdf"]],
            database=TEST_DB,
            debug=True,
        )
    print_meta("large_pdf (Files API + split)", m)
    print(f"  text = {r.get('text','')[:150]}")

  [large_pdf 첫 호출 (업로드+split+LLM)] 18675ms
--- large_pdf (Files API + split) ---
  source        = live
  cache_hit     = False
  cache_key     = ee9065570b2c3dd9...
  route_reason  = cache_miss
  text = The Korean title from the image is: "2025 GS건설 지속가능경영보고서".

In English, this translates to: "2025 GS E&C Sustainability Report".

**Summary:**
This do


## 14. 부분 섞기 — inline + Files API 혼재

`small_pdf` + `large_pdf` → 작은 파일은 inline, 큰 파일은 Files API로 분류.

In [19]:
if "large_pdf" not in inputs:
    print("  [SKIP] large_file.pdf 없음")
else:
    r, m = await llm_request_async(
        {"contents": unique_q("두 PDF의 공통점 한 줄"), "model": "gemini-2.5-flash"},
        files=[inputs["small_pdf"], inputs["large_pdf"]],
        database=TEST_DB,
        debug=True,
    )
    print_meta("small + large (혼합)", m)
    print(f"  text = {r.get('text','')[:150]}")

--- small + large (혼합) ---
  source        = live
  cache_hit     = False
  cache_key     = 5b9afbdb4df2bf7b...
  route_reason  = cache_miss
  text = 두 PDF의 가장 중요한 공통점은 **ESG 경영 활동 및 성과 보고**입니다.
(Both PDF documents are essentially **reporting on ESG (Environmental, Social, and Governance) management


## 15. 이미지+PDF 혼합 — 20MB 한도 분기

이미지/오디오가 포함되면 inline 합계 한도가 20MB로 낮아진다. `image(1.9MB) + medium_pdf(24MB)` → 합계 26MB → 큰 쪽이 Files API로 승급.

In [20]:
if "medium_pdf" not in inputs:
    print("  [SKIP] medium_file.pdf 없음")
else:
    r, m = await llm_request_async(
        {"contents": unique_q("이미지와 PDF를 함께 설명"), "model": "gemini-2.5-flash"},
        files=[inputs["image"], inputs["medium_pdf"]],
        database=TEST_DB,
        debug=True,
    )
    print_meta("image + medium_pdf (20MB 한도)", m)
    print(f"  text = {r.get('text','')[:150]}")

--- image + medium_pdf (20MB 한도) ---
  source        = live
  cache_hit     = False
  cache_key     = 51438ca3f65d3cb8...
  route_reason  = cache_miss
  text = The provided document is the **DB HiTek ESG Report for 2024-2025**.

**Textual Analysis:**
*   **Company Name:** "DB HiTek" (English) and "DB하이텍" (Kor


## 16. `context_cache_ttl` — Context Cache 생성

Files API로 분류된 파일에 TTL(초)을 지정하면 Gemini Context Cache 생성. 같은 파일 조합으로 반복 호출 시 입력 토큰 90% 할인.

In [21]:
if "large_pdf" not in inputs:
    print("  [SKIP] large_file.pdf 없음")
else:
    # context_cache 생성/재사용 여부는 서비스 로그에서 확인:
    #   첫 호출: [route] mode=context_cache source=live  name=cachedContents/...
    #   두번째:  [route] mode=context_cache source=cached name=cachedContents/...
    r, m = await llm_request_async(
        {"contents": unique_q("첫 질문"), "model": "gemini-2.5-flash"},
        files=[inputs["large_pdf"]],
        timings={"context_cache_ttl": 3600},
        database=TEST_DB,
        debug=True,
    )
    print_meta("context_cache 첫 호출 (생성)", m)
    assert m.get("source") == "live", "첫 호출은 LLM 직접 호출이어야 함"

    r2, m2 = await llm_request_async(
        {"contents": unique_q("두번째 질문"), "model": "gemini-2.5-flash"},
        files=[inputs["large_pdf"]],
        timings={"context_cache_ttl": 3600},
        database=TEST_DB,
        debug=True,
    )
    print_meta("context_cache 두번째 호출 (재사용)", m2)
    assert m2.get("source") == "live", "새 질문이므로 MDB miss, LLM 직접 호출"
    print("  OK: context_cache 재사용은 서비스 로그 [route] mode=context_cache source=cached 로 확인")

--- context_cache 첫 호출 (생성) ---
  source        = live
  cache_hit     = False
  cache_key     = aeff6e70cdf60715...
  route_reason  = cache_miss
--- context_cache 두번째 호출 (재사용) ---
  source        = live
  cache_hit     = False
  cache_key     = 47e65baedfbdc225...
  route_reason  = cache_miss
  OK: context_cache 재사용은 서비스 로그 [route] mode=context_cache source=cached 로 확인


## 17. 검증 에러 — 지원 안 되는 mime 타입

`test_audio.m4a` (audio/mp4 또는 audio/x-m4a) → 화이트리스트 미포함 → HTTP 422.

In [22]:
from llm_request import ServiceError

if "audio" not in inputs:
    print("  [SKIP] test_audio.m4a 없음")
else:
    print(f"  파일 mime: {inputs['audio'].mime_type}")
    try:
        r, m = await llm_request_async(
            {"contents": "전사해줘", "model": "gemini-2.5-flash"},
            files=[inputs["audio"]],
            database=TEST_DB,
        )
        print("  [FAIL] 예외 발생 안 함")
    except ServiceError as e:
        print(f"  [OK] ServiceError: status={e.status_code} type={e.error_type}")
        print(f"       message: {str(e)[:120]}")

  파일 mime: audio/mp4
  [OK] ServiceError: status=422 type=unsupported_file_type
       message: [unsupported_file_type] 지원하지 않는 파일 타입: mime_type=audio/mp4, file_name=test_audio.m4a (422)


## 18. 동시성 — single-flight dedup

동일 입력 3개를 동시에 호출하면 첫 번째만 LLM 호출, 나머지는 결과를 공유받는다.

In [23]:
import asyncio

Q_D = unique_q("동시성 테스트 질문")


async def _one(i: int):
    r, m = await llm_request_async(
        {"contents": Q_D, "model": "gemini-2.5-flash"},
        database=TEST_DB,
        debug=True,
    )
    return i, r.get("text", "")[:40], m


with timer("동시 3개 요청"):
    results = await asyncio.gather(_one(1), _one(2), _one(3))

for i, txt, meta in results:
    route = meta.get("route_reason")
    print(f"  #{i}: cache_hit={meta.get('cache_hit')!s:<5} route={route!r:<30} text={txt}")

texts = {txt for _, txt, _ in results}
print(f"\n  고유 응답 수: {len(texts)} (1이면 dedup 동작)")

  [동시 3개 요청] 25585ms
  #1: cache_hit=False route='cache_miss'                   text=동시성 테스트(Concurrency Testing)는 여러 개의 스레드나
  #2: cache_hit=True  route='standby_wait_hit'             text=동시성 테스트(Concurrency Testing)는 여러 개의 스레드나
  #3: cache_hit=True  route='standby_wait_hit'             text=동시성 테스트(Concurrency Testing)는 여러 개의 스레드나

  고유 응답 수: 1 (1이면 dedup 동작)


## 정리

위 셀들이 모두 성공하면 다음이 검증된 것:

- 공개 함수 6종 전부 정상 동작
- 캐시 경로: miss → hit, 우회(cache_mode=False), 저장 스킵(save_mode=False)
- 파일 분기: inline(PDF/이미지/텍스트), Files API, 부분 섞기, 20MB 한도 분기, PDF split
- context_cache 생성/재사용
- 입력 검증(mime) fail-fast
- single-flight dedup

실패한 셀이 있으면 §2 헬스체크부터 재확인.